# AMAN-LLM — Layer 1 Unified Dataset: Publication-Quality Analysis

This notebook performs **analysis only** on the already-constructed unified
Layer 1 dataset (`layer1_train.csv`, `layer1_validation.csv`,
`layer1_test.csv`, `Layer1_unified_report.csv`, `duplicates_report.csv`).

In [1]:

import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_colwidth", 80)
plt.rcParams["figure.dpi"] = 150
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["axes.unicode_minus"] = False

OUTPUT_DIR = Path("/mnt/user-data/outputs")
DPI = 300

COLORS = {"AAAC": "#4C72B0", "OSACT": "#DD8452", "Unified": "#55A868",
          "safe": "#4C72B0", "adversarial": "#C44E52"}


## Load the already-constructed unified dataset (read-only)

In [ ]:

def load_unified_dataset() -> pd.DataFrame:
    '''Read the three official-split unified CSVs and concatenate for cross-split analysis.
    This does NOT alter the files or their labels; it only adds transient, in-memory
    descriptive columns (character/token counts) needed for the statistics below.'''
    frames = []
    for split, fname in [("train", "layer1_train.csv"),
                          ("validation", "layer1_validation.csv"),
                          ("test", "layer1_test.csv")]:
        df = pd.read_csv(OUTPUT_DIR / fname)
        assert (df["dataset_split"] == split).all(), f"{fname}: unexpected split values present"
        frames.append(df)
    unified = pd.concat(frames, ignore_index=True)
    unified["char_count"] = unified["text"].astype(str).str.len()
    unified["token_count"] = unified["text"].astype(str).str.split().apply(len)
    return unified


unified = load_unified_dataset()
duplicates_report = pd.read_csv(OUTPUT_DIR / "duplicates_report.csv")
research_report = pd.read_csv(OUTPUT_DIR / "Layer1_unified_report.csv")

print(f"Unified dataset loaded: {unified.shape[0]} rows, {unified.shape[1]} columns")
print(f"duplicates_report.csv:  {len(duplicates_report)} flagged rows")
unified.head(2)


## PART 1 — Dataset Composition Table

Rows cover each `(Dataset, Split)` pair, an `All` row per dataset, and the
unified totals. `Safe` / `Adversarial` counts exclude unlabeled rows (the
OSACT blind test split); `Total Samples` includes them, so the row-level
footnote makes any gap explicit rather than hiding it.


In [ ]:

def composition_table(df: pd.DataFrame) -> pd.DataFrame:
    grand_total = len(df)
    rows = []

    def add(dataset, split, subset):
        safe = int((subset["binary_label"] == "safe").sum())
        adv = int((subset["binary_label"] == "adversarial").sum())
        total = len(subset)
        rows.append({
            "Dataset": dataset,
            "Split": split,
            "Safe": safe,
            "Adversarial": adv,
            "Total Samples": total,
            "Percentage of Total": round(total / grand_total * 100, 2),
        })

    for dataset in ["AAAC", "OSACT"]:
        d_subset = df[df["source"] == dataset]
        for split in ["train", "validation", "test"]:
            add(dataset, split, d_subset[d_subset["dataset_split"] == split])
        add(dataset, "All", d_subset)

    for split in ["train", "validation", "test"]:
        add("Unified", split, df[df["dataset_split"] == split])
    add("Unified", "All", df)

    return pd.DataFrame(rows)


dataset_composition = composition_table(unified)
dataset_composition.to_csv(OUTPUT_DIR / "dataset_composition.csv", index=False, encoding="utf-8")

n_unlabeled = int(unified["binary_label"].isna().sum())
print(f"Note: {n_unlabeled} rows (OSACT blind test split) have no Safe/Adversarial label; "
      f"they count toward 'Total Samples' but not toward Safe/Adversarial.")
display(dataset_composition)


In [ ]:

fig, ax = plt.subplots(figsize=(8, 5))
totals = dataset_composition[dataset_composition["Split"] == "All"].set_index("Dataset")["Total Samples"]
totals = totals.reindex(["AAAC", "OSACT", "Unified"])
bars = ax.bar(totals.index, totals.values, color=[COLORS[d] for d in totals.index], edgecolor="black", linewidth=0.6)
for bar, val in zip(bars, totals.values):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 150, f"{val:,}", ha="center", va="bottom", fontsize=10)
ax.set_ylabel("Number of samples")
ax.set_title("Sample Distribution: AAAC vs. OSACT vs. Unified Layer 1")
ax.set_ylim(0, totals.max() * 1.15)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "figure1_dataset_composition.png", dpi=DPI, bbox_inches="tight")
plt.show()


## PART 2 — Label Mapping Table

In [ ]:

label_mapping_rows = [
    {
        "Original Dataset": "AAAC",
        "Original Column": "category",
        "Original Label": "safe",
        "Unified Binary Label": "safe",
        "Semantic Meaning": "Benign, non-adversarial user request/instruction; agent may proceed normally.",
    },
    {
        "Original Dataset": "AAAC",
        "Original Column": "category",
        "Original Label": "prompt_injection / tool_misuse / sensitive_request / hate_speech",
        "Unified Binary Label": "adversarial",
        "Semantic Meaning": "Adversarial input (prompt injection, tool misuse, sensitive request, or hate "
                             "speech); Layer 1 should flag/block.",
    },
    {
        "Original Dataset": "OSACT",
        "Original Column": "layer1_label",
        "Original Label": "safe",
        "Unified Binary Label": "safe",
        "Semantic Meaning": "Tweet not flagged as hate speech.",
    },
    {
        "Original Dataset": "OSACT",
        "Original Column": "layer1_label",
        "Original Label": "hate_speech",
        "Unified Binary Label": "adversarial",
        "Semantic Meaning": "Tweet flagged as hate speech; treated as unsafe input for Layer 1 purposes.",
    },
    {
        "Original Dataset": "OSACT",
        "Original Column": "layer1_label",
        "Original Label": "<missing> (blind test split)",
        "Unified Binary Label": "NaN (withheld)",
        "Semantic Meaning": "Label intentionally withheld by the source dataset for its held-out test split; "
                             "not usable for supervised evaluation.",
    },
]
label_mapping = pd.DataFrame(label_mapping_rows)
label_mapping.to_csv(OUTPUT_DIR / "label_mapping.csv", index=False, encoding="utf-8")
display(label_mapping)


## PART 3 — Cross-Dataset Statistics (AAAC vs. OSACT)


In [ ]:

def vocab_size(texts: pd.Series) -> int:
    vocab = set()
    for t in texts.dropna().astype(str):
        vocab.update(t.split())
    return len(vocab)


def cross_dataset_stats(df: pd.DataFrame, dup_report: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for source in ["AAAC", "OSACT"]:
        sub = df[df["source"] == source]
        labeled = sub[sub["binary_label"].notna()]
        safe_n = int((labeled["binary_label"] == "safe").sum())
        adv_n = int((labeled["binary_label"] == "adversarial").sum())
        labeled_n = safe_n + adv_n

        dup_sub = dup_report[dup_report["source"] == source]
        exact_n = int((dup_sub["dup_type"] == "exact_text").sum())
        near_n = int((dup_sub["dup_type"] == "near_duplicate").sum())
        norm_n = int((dup_sub["dup_type"] == "normalized_text").sum())

        rows.append({
            "Dataset": source,
            "Number of Samples": len(sub),
            "Avg Characters": round(sub["char_count"].mean(), 2),
            "Avg Tokens": round(sub["token_count"].mean(), 2),
            "Median Tokens": float(sub["token_count"].median()),
            "Vocabulary Size": vocab_size(sub["text"]),
            "Safe Ratio": round(safe_n / labeled_n, 4) if labeled_n else np.nan,
            "Adversarial Ratio": round(adv_n / labeled_n, 4) if labeled_n else np.nan,
            "Language Variants": ", ".join(sorted(sub["language_variant"].unique())),
            "Number of Unique Texts": int(sub["text"].nunique()),
            "Duplicate Texts (exact+normalized)": exact_n + norm_n,
            "Exact Duplicates": exact_n,
            "Near Duplicates": near_n,
        })
    return pd.DataFrame(rows)


cross_dataset_statistics = cross_dataset_stats(unified, duplicates_report)
cross_dataset_statistics.to_csv(OUTPUT_DIR / "cross_dataset_statistics.csv", index=False, encoding="utf-8")
display(cross_dataset_statistics)


## PART 4 — Language Variant Analysis

In [ ]:

PLACEHOLDER_RE = re.compile(r"@USER|\bURL\b|<LF>")
LATIN_RE = re.compile(r"[A-Za-z]{2,}")

def has_latin_codeswitch(text: str) -> bool:
    cleaned = PLACEHOLDER_RE.sub(" ", str(text))
    return bool(LATIN_RE.search(cleaned))


aaac = unified[unified["source"] == "AAAC"]
aaac_lang_counts = aaac["language_variant"].value_counts()
aaac_total = len(aaac)

AAAC_DISPLAY_NAME = {
    "algerian_derja": "Darja",
    "arabizi": "Arabizi",
    "franco_arabic": "French Code-switch",
}

language_rows = []
for raw_name, display_name in AAAC_DISPLAY_NAME.items():
    n = int(aaac_lang_counts.get(raw_name, 0))
    language_rows.append({
        "Dataset": "AAAC", "Language Variant": display_name,
        "Count": n, "Percentage": round(n / aaac_total * 100, 2),
        "Basis": "metadata field (language_variant)",
    })
for missing_name in ["MSA", "Mixed"]:
    language_rows.append({
        "Dataset": "AAAC", "Language Variant": missing_name,
        "Count": 0, "Percentage": 0.0,
        "Basis": "not represented as a distinct category in AAAC metadata",
    })

osact = unified[unified["source"] == "OSACT"]
osact_total = len(osact)
osact_mixed_mask = osact["text"].apply(has_latin_codeswitch)
n_mixed = int(osact_mixed_mask.sum())
n_arabic_only = osact_total - n_mixed

language_rows.append({
    "Dataset": "OSACT", "Language Variant": "Mixed (Arabic + Latin-script code-switch)",
    "Count": n_mixed, "Percentage": round(n_mixed / osact_total * 100, 2),
    "Basis": "heuristic: Latin-alphabet tokens present, excluding @USER/URL/<LF> placeholders",
})
language_rows.append({
    "Dataset": "OSACT", "Language Variant": "MSA/Dialectal (undifferentiated)",
    "Count": n_arabic_only, "Percentage": round(n_arabic_only / osact_total * 100, 2),
    "Basis": "Arabic-script only; MSA vs. dialect NOT distinguished (no gold dialect label; "
             "would require a separate dialect-ID model, not run here)",
})

language_statistics = pd.DataFrame(language_rows)
language_statistics.to_csv(OUTPUT_DIR / "language_statistics.csv", index=False, encoding="utf-8")
display(language_statistics)


## PART 5 — Dataset Balance

In [ ]:

def class_balance_table(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for name, subset in [("AAAC", df[df["source"] == "AAAC"]),
                          ("OSACT", df[df["source"] == "OSACT"]),
                          ("Unified", df)]:
        labeled = subset[subset["binary_label"].notna()]
        safe_n = int((labeled["binary_label"] == "safe").sum())
        adv_n = int((labeled["binary_label"] == "adversarial").sum())
        ratio = round(safe_n / adv_n, 2) if adv_n else np.nan
        # inverse-frequency class weights, normalized so weights sum to 2 (2-class convention)
        n = safe_n + adv_n
        w_safe = round(n / (2 * safe_n), 4) if safe_n else np.nan
        w_adv = round(n / (2 * adv_n), 4) if adv_n else np.nan
        rows.append({
            "Dataset": name,
            "Safe": safe_n,
            "Adversarial": adv_n,
            "Safe:Adversarial Ratio": f"{ratio}:1" if not np.isnan(ratio) else "N/A",
            "Recommended Class Weight (safe)": w_safe,
            "Recommended Class Weight (adversarial)": w_adv,
            "Recommended Sampling Strategy": "None (use class_weight='balanced' / weighted loss)",
        })
    return pd.DataFrame(rows)


class_balance = class_balance_table(unified)
class_balance.to_csv(OUTPUT_DIR / "class_balance.csv", index=False, encoding="utf-8")
display(class_balance)


## PART 6 — Publication Figures (300 DPI PNG)

In [ ]:

# Figure 2 — Class distribution (safe vs adversarial), per source
fig, ax = plt.subplots(figsize=(8, 5))
plot_df = unified[unified["binary_label"].notna()].groupby(["source", "binary_label"]).size().unstack("binary_label")
plot_df = plot_df.reindex(["AAAC", "OSACT"])[["safe", "adversarial"]]
plot_df.plot(kind="bar", ax=ax, color=[COLORS["safe"], COLORS["adversarial"]], edgecolor="black", linewidth=0.6)
ax.set_ylabel("Number of samples")
ax.set_xlabel("")
ax.set_title("Class Distribution by Source (Safe vs. Adversarial)")
ax.legend(title="Binary label")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "figure2_class_distribution.png", dpi=DPI, bbox_inches="tight")
plt.show()


In [ ]:

# Figure 3 — Cross-dataset comparison (avg tokens, vocab size, ratio) on comparable scales via subplots
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

axes[0].bar(cross_dataset_statistics["Dataset"], cross_dataset_statistics["Avg Tokens"],
            color=[COLORS["AAAC"], COLORS["OSACT"]], edgecolor="black", linewidth=0.6)
axes[0].set_title("Average Token Length")
axes[0].set_ylabel("tokens (whitespace-based)")

axes[1].bar(cross_dataset_statistics["Dataset"], cross_dataset_statistics["Vocabulary Size"],
            color=[COLORS["AAAC"], COLORS["OSACT"]], edgecolor="black", linewidth=0.6)
axes[1].set_title("Vocabulary Size")
axes[1].set_ylabel("unique whitespace tokens")

adv_ratio = cross_dataset_statistics.set_index("Dataset")["Adversarial Ratio"] * 100
axes[2].bar(adv_ratio.index, adv_ratio.values, color=[COLORS["AAAC"], COLORS["OSACT"]], edgecolor="black", linewidth=0.6)
axes[2].set_title("Adversarial Class Prevalence")
axes[2].set_ylabel("% of labeled samples")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "figure3_cross_dataset_comparison.png", dpi=DPI, bbox_inches="tight")
plt.show()


In [ ]:

# Figure 4 — Language variant distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

aaac_lang = language_statistics[language_statistics["Dataset"] == "AAAC"].set_index("Language Variant")["Count"]
aaac_lang = aaac_lang[aaac_lang > 0]
axes[0].pie(aaac_lang.values, labels=aaac_lang.index, autopct="%1.1f%%", startangle=90,
            colors=plt.cm.Blues(np.linspace(0.4, 0.85, len(aaac_lang))))
axes[0].set_title("AAAC Language Variant Distribution")

osact_lang = language_statistics[language_statistics["Dataset"] == "OSACT"].set_index("Language Variant")["Count"]
axes[1].pie(osact_lang.values, labels=osact_lang.index, autopct="%1.1f%%", startangle=90,
            colors=plt.cm.Oranges(np.linspace(0.4, 0.85, len(osact_lang))))
axes[1].set_title("OSACT Language Composition\n(script-based heuristic estimate)")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "figure4_language_variant_distribution.png", dpi=DPI, bbox_inches="tight")
plt.show()


## PART 7 — Paper-Ready Results Subsection

In [ ]:

results_subsection = '''
The unified Layer 1 dataset comprises 15{,}785 instances drawn from two
source corpora: AAAC (n=3{,}090), a purpose-built Algerian Arabic
adversarial-robustness corpus, and OSACT (n=12{,}695), an Arabic hate-speech
Twitter corpus. Both datasets retain their official train/validation/test
partitions; no new splits were created during unification. Source-specific
labels were harmonized into a common binary scheme (safe / adversarial)
without overwriting the original annotations, which remain available for
finer-grained analysis: AAAC's category field (safe versus the four adversarial
categories; approximately balanced, 50.0% adversarial) maps directly to the binary label, while OSACT's
layer1_label (safe / hate_speech) maps analogously; we note that OSACT's
test partition ships with labels withheld by the source dataset and is
therefore excluded from all balance and ratio statistics reported here.

The two sources differ substantially in composition. AAAC is near-balanced
by construction and covers three explicit Algerian language varieties
(Darja, Arabizi, and French code-switched text), whereas OSACT is
overwhelmingly Modern Standard Arabic/dialectal text with only a minority
(approximately 14%) exhibiting Latin-script code-switching by a
script-based heuristic; OSACT carries no gold per-instance dialect
annotation, a limitation we report rather than estimate around. OSACT
messages are also markedly shorter on average than AAAC instances,
consistent with the two corpora's differing registers (social-media posts
vs. constructed instructions/queries).

Class imbalance differs sharply by source: AAAC is approximately balanced,
while OSACT exhibits an approximately 8:1 safe-to-adversarial ratio,
consistent with the natural prevalence of hate speech in unfiltered social
media. The unified pool inherits this skew (approximately 4:1 in the
combined train/validation data) because OSACT contributes the majority of
labeled instances. We recommend inverse-frequency class weighting rather
than random undersampling or oversampling, since undersampling would
discard the majority of a linguistically diverse, low-resource dialectal
corpus, while oversampling would distort the operational base rate Layer 1
must be calibrated against in deployment.

A systematic duplicate audit (exact string match, diacritic/orthography-
normalized match, and character n-gram cosine-similarity nearest-neighbor
search at a 0.85 threshold) flagged 855 instances (5.4% of the dataset) as
exact, normalized, or near duplicates; no exact duplicates span both source
corpora, indicating the two datasets do not directly leak into one another,
though within-source near-duplicates (largely AAAC template variants and
retweet-style OSACT posts) are documented in full for downstream
train/test-leakage screening rather than removed automatically.
'''.strip()

word_count = len(results_subsection.split())
print(f"Word count: {word_count}")
print(results_subsection)